## PIP INSTALL

In [1]:
%pip install requests
%pip install deltalake
%pip install pyarrow
%pip install spotipy
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### IMPORT


In [1]:
from datetime import datetime
import pandas as pd
import os
from dotenv import load_dotenv
from string import Template
# FUCIONES
from utils import get_data, upsert_data_as_delta, build_table 

load_dotenv()
client_id = os.getenv('clientID')
api_key = os.getenv('apiKEY')
FM_API_KEY = os.getenv('FM_API_KEY')
FM_API_KEY_FORMATJSON = f"&api_key={FM_API_KEY}&format=json"

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

# Spotipy
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(client_id=client_id, client_secret=api_key))

#FM Api URL
base_url_fm = "http://ws.audioscrobbler.com/2.0/"
geo_endpoint_fm =f"?method=geo.gettopartists&country=argentina&limit=100"


artist_top_track = Template('?method=artist.gettoptracks&mbid=${mbid}&limit=1')
FECHA_EXTRACCION = datetime.now().strftime("%Y-%m-%d")

### Top 100 artistas de argentina.

In [2]:
artists = get_data(base_url_fm, geo_endpoint_fm,FM_API_KEY_FORMATJSON,data_field='topartists')
all_artists = []
rank = 1
for i in artists['artist']:
  artist = {}
  artist['FM_artist_mbid']=i['mbid']
  artist['FM_artist_name']=i['name']
  artist['FM_rank']=rank
  top_track = get_data(base_url_fm,artist_top_track.substitute(mbid=i['mbid']) ,FM_API_KEY_FORMATJSON,data_field='toptracks')
  for j in top_track['track']:
    artist['FM_top_track_name'] = j['name']
    try:
      if j['mbid']:
       artist['FM_top_track_mbid'] = j['mbid']
    except:
      artist['FM_top_track_mbid'] = 'N/A'
    artist['FM_top_track_listeners'] = j['listeners']
    artist['FM_top_track_playcount'] = j['playcount']
    spotify_track = sp.search(q=f"track:{j['name']} artist:{i['name']}",limit=1)
    try:
      artist['SP_track_id'] = spotify_track['tracks']['items'][0]['id']
      artist['SP_album_release_date'] = spotify_track['tracks']['items'][0]['album']['release_date']
      artist['SP_track_popularity'] = spotify_track['tracks']['items'][0]['popularity']
    except:
      artist['SP_track_id']  = 'N/A'
      artist['SP_album_release_date'] = 'N/A'
      artist['SP_track_popularity'] = 'N/A'
  artist['date_extraction'] = FECHA_EXTRACCION
  rank = rank + 1
  all_artists.append(artist)

## BUILD DATAFRAME

In [3]:
df_artists = build_table(all_artists)

df_artists["date_extraction"] = pd.to_datetime(df_artists["date_extraction"])



df_artists

,FM_artist_mbid,FM_artist_name,FM_rank,FM_top_track_name,FM_top_track_mbid,FM_top_track_listeners,FM_top_track_playcount,SP_track_id,SP_album_release_date,SP_track_popularity,date_extraction
0,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,1,Creep,d11fcceb-dfc5-4d19-b45d-f4e8f6d3eaa6,2977581,35530801,70LcF31zb1H0PyJoS1Sx1r,1993-02-22,87,2024-09-13
1,5441c29d-3602-4898-b1a1-b77fa23b8e50,David Bowie,2,Starman - 2012 Remaster,N/A,788919,7601887,0pQskrTITgmCMyr85tb9qq,1972-06-06,78,2024-09-13
2,420ca290-76c5-41af-999e-564d7c71f1a7,Queen,3,Bohemian Rhapsody,ecfa6746-7bc4-4088-ace6-d209477bd63f,1872550,12942114,7tFiyTwD0nx5a1eklYtX2J,1975-11-21,71,2024-09-13
3,8bfac288-ccc5-448d-9573-c33ea2aa5c30,Red Hot Chili Peppers,4,Californication,084a24a9-b289-4584-9fb5-1ca0f7500eb3,2429906,20523278,48UPSzbZjgc449aqz8bxox,1999-06-08,83,2024-09-13
4,b10bbbfc-cf9e-42e0-be17-e2c3e1d2600d,The Beatles,5,Eleanor Rigby,424c10aa-a857-4dc6-872e-2a8fb0b707f3,1091482,7358638,5GjPQ0eI7AgmOnADn1EO6Q,1966-08-05,72,2024-09-13
...,...,...,...,...,...,...,...,...,...,...,...
95,7364dea6-ca9a-48e3-be01-b44ad0d19897,a-ha,96,Take on Me,c9bf13e6-619e-4cca-ae0f-512e9a478930,2246877,17304818,2WfaOiMkCvy7F5fcp2zZ8L,1985-06-01,88,2024-09-13
96,8970d868-0723-483b-a75b-51088913d3d4,Moby,97,Porcelain,a337c5df-7286-44d7-9296-8bc1d8549bd7,1099204,7873939,19vUJwIaPAgDJyGZMBI9MG,1999-05-17,55,2024-09-13
97,06fb1c8b-566e-4cb2-985b-b467c90781d4,Jimi Hendrix,98,All Along the Watchtower,81edf1da-d812-4e08-96dc-bf8c48edef1a,1478117,10889307,2aoo2jlRnM3A0NyLQqMN2f,1968-10-25,74,2024-09-13
98,ff401398-6d43-4e91-8f87-33fe19aafd84,Pescado Rabioso,99,Cementerio Club,9bf5ce0b-0725-4e4d-ac6a-ba74e298fd17,99025,942455,0O9BqfU1Cdm7YqmD0TrT9i,1973-05-07,56,2024-09-13


## DELTALAKE BRONZE

In [6]:

bronze_dir = "datalake/bronze/FM_SP_api"
raw_dir = f"./{bronze_dir}"
upsert_data_as_delta(df_artists,raw_dir,"target.SP_track_id = source.SP_track_id")

TableNotFoundError: Could not create local directory: C:/Users/Valentin%20Kebat/Desktop/CURSO%20DATA%20UTN/TP%20FINAL/datalake/bronze/FM_SP_api/
Error: Os { code: 5, kind: PermissionDenied, message: "Acceso denegado." }

In [35]:
## Registro todos los ids que forman parte de la extracción actual
from utils import save_new_data_as_delta
from pathlib import Path
bronze_dir_top_100 = Path("./datalake/bronze/FM_SP_api_top_100")
df_artists_top_100 = df_artists[['SP_track_id', 'date_extraction']].copy()

# Asegúrate de que la columna es de tipo datetime
df_artists_top_100.loc[:, 'date_extraction'] = pd.to_datetime(df_artists_top_100['date_extraction'])

df_artists_top_100.loc[:, 'date_extraction'] = df_artists_top_100['date_extraction'].dt.date

#df_artists_top_100.loc[:, 'date_extraction'] = df_artists_top_100['date_extraction'] + pd.Timedelta(days=4)
save_new_data_as_delta(df_artists_top_100,bronze_dir_top_100,predicate="target.SP_track_id = source.SP_track_id", partition_cols=['date_extraction'])
df_artists_top_100

,SP_track_id,date_extraction
0,70LcF31zb1H0PyJoS1Sx1r,2024-09-12
2,7tFiyTwD0nx5a1eklYtX2J,2024-09-12
4,5GjPQ0eI7AgmOnADn1EO6Q,2024-09-12
5,2lpIh6Gr6HYjg1CFBaucS5,2024-09-12
6,6H3kDe7CGoWYBabAeVWGiD,2024-09-12
10,263aNAQCeFSWipk896byo6,2024-09-12
11,5ghIJDpPoe3CfHMGu71E6T,2024-09-12
12,1FTSo4v6BOZH9QxKc3MbVM,2024-09-12
13,0d28khcov6AiegSCpG5TuT,2024-09-12
14,6mFkJmJqdDVQ1REhVfGgd1,2024-09-12
